# ASR Roundtrip: Are the TTS Words Recognized Correctly?

In [ ]:
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec peft

## Load the Generated TTS Sentences

In [ ]:
from pathlib import Path
import pandas as pd
import torchaudio

problem_words = pd.read_csv("problem_words.csv", sep="\t")

TTS_DIR = Path("generated/kitten")

tts_samples = []
for w in problem_words["word"].head(20):
    wav_path = TTS_DIR / f"{w}.wav"
    if wav_path.exists():
        tts_samples.append({"word": w, "audio": wav_path.as_posix()})

print(f"{len(tts_samples)} TTS samples loaded")
tts_samples[:3]

## Roundtrip Inference

In [ ]:
from tqdm.auto import tqdm

def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)
    waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
    return waveform.mean(dim=0)

rows = []
for sample in tqdm(tts_samples):
    wav = load_audio(sample["audio"])
    hyp_granite = transcribe_granite(wav)
    rows.append({**sample,
                 "granite_hypothesis": hyp_granite,
})

roundtrip = pd.DataFrame(rows)
roundtrip.head()

## Word Recognition Analysis

In [ ]:
import re
import string

def normalize(text):
    text = str(text).translate(str.maketrans("", "", string.punctuation)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text

def word_recognized(word, hypothesis):
    """A word counts as recognized if it appears (normalized) in the hypothesis."""
    return normalize(word) in normalize(hypothesis).split()

for asr_col in ["granite_hypothesis"]:
    asr_name = asr_col.replace("_hypothesis", "")
    roundtrip[f"{asr_name}_recognized"] = [
        word_recognized(w, h) for w, h in zip(roundtrip["word"], roundtrip[asr_col])
    ]

roundtrip.head()

## Recognition Rate

In [ ]:
recognition_rate = roundtrip["granite_recognized"].mean()
print(f"Granite recognized {recognition_rate:.0%} of the TTS words")

In [ ]:
per_word = roundtrip.set_index("word")[["granite_hypothesis", "granite_recognized"]]
per_word

## Qualitative Inspection

In [ ]:
from IPython.display import Audio, display

for _, row in roundtrip.iterrows():
    if not row["granite_recognized"]:
        print("=" * 70)
        print("WORD :", row["word"])
        print("GRANITE :", row["granite_hypothesis"])
        display(Audio(row["audio"]))

## Save the Roundtrip Results

In [ ]:
roundtrip.to_csv("roundtrip_results.tsv", sep="\t", index=False)
summary.to_csv("roundtrip_summary.tsv", sep="\t")
print("Saved roundtrip_results.tsv and roundtrip_summary.tsv")